# SmartSignal Replacement — AAKR Anomaly Detection
**Auto-Associative Kernel Regression (AAKR)** — the same algorithm used by GE SmartSignal — implemented natively in Fabric PySpark.
For each sensor at each time step, AAKR answers: *"Given what all other correlated sensors are reading right now, what should this sensor read?"* The residual (actual - expected) is the anomaly signal.
**Pipeline:**
1. Build training memory from known-good PI data (running, no outages, no SmartSignal alerts)
2. AAKR scoring engine: generate expected values + residuals for every tag
3. Threshold calibration using SmartSignal incidents as ground truth
4. Alert/incident generation with episode grouping + severity
5. Validation: side-by-side comparison with SmartSignal
**Advantages over SmartSignal:**
- Runs entirely in Fabric — no vendor dependency, no WCF SDK, no Windows server
- Covers ALL 308 PI tags (SmartSignal only monitors 60)
- Transparent, auditable thresholds
- Integrated with the OneGrid gold tables


In [ ]:
import uuid
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta
notebook_run_id = str(uuid.uuid4())
# ─── Target Assets ───
TARGET_ASSETS = ['RV2_U2_Boiler', 'RV3_U3_Steam_Turbine', 'RV3_U3_Boiler_Feed_Pump_East']
# ─── AAKR Hyperparameters ───
MEMORY_SAMPLE_SIZE = 2000      # Training memory rows per asset (stratified by MW bin)
KERNEL_BANDWIDTH = 'auto'      # 'auto' = median pairwise distance x 0.5
RESIDUAL_ALERT_SIGMA = 3.0     # |z_residual| threshold for anomaly
MIN_COMPLETENESS = 0.80        # Minimum fraction of tags present to score a bin
EXCLUSION_BUFFER_HOURS = 6     # Hours before/after events to exclude from training
# ─── Baseline / Scoring Windows ───
BASELINE_DAYS = 90
BIN_SECONDS = 900              # 15-minute bins
SCORING_LOOKBACK_HOURS = 4
EPISODE_GAP_MIN = 60
# ─── Running State Tags ───
RUNNING_STATE = {
    'RV2_U2_Boiler':               {'tag': 'RV2:BTPU2BPDRUM.AG', 'threshold': 600, 'op': '>'},
    'RV3_U3_Steam_Turbine':        {'tag': 'RV3:TXSU3TS15A.AG',  'threshold': 6,   'op': '>'},
    'RV3_U3_Boiler_Feed_Pump_East': {'tag': 'RV3:FPSU3TS24E.AG', 'threshold': 20,  'op': '>'}
}
# ─── MW Load Tags (for load conditioning) ───
MW_TAGS = {
    'RV2_U2_Boiler':               'RV2:GEJU2GE03.AG',
    'RV3_U3_Steam_Turbine':        'RV3:GEJU3GE03.AG',
    'RV3_U3_Boiler_Feed_Pump_East': 'RV3:GEJU3GE03.AG'
}
MW_BIN_WIDTH = 50
# ─── Tables ───
PI_TABLE = "gold.fact_pi"
BRIDGE_TABLE = "gold.bridge_pi_tag_to_asset"
MEMORY_TABLE = "ml.aakr_memory"
MEMORY_META_TABLE = "ml.aakr_metadata"
SCORED_TABLE = "ml.aakr_scores"
EPISODES_TABLE = "ml.aakr_episodes"
HEALTH_TABLE = "ml.aakr_health"
# ─── Eventhouse ───
KUSTO_URI = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
KUSTO_DB = "pi-realtime-db"
def _kusto_tok():
    try:
        import notebookutils as _n; _c = _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; _c = _m.credentials
    for _a in (KUSTO_URI, "kusto", "pbi"):
        try:
            _t = _c.getToken(_a)
            if _t: return _t
        except Exception:
            pass
    raise RuntimeError("could not acquire Kusto token")

def read_kusto(query):
    return (spark.read
        .format("com.microsoft.kusto.spark.datasource")
        .option("accessToken", _kusto_tok())
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .load())
def safe_col(tag):
    """Sanitize PI tag name for use as Spark column name."""
    return tag.replace(":", "_").replace(".", "_").replace("-", "_").replace(" ", "_")
def unsafe_col(safe):
    """Reverse not possible uniquely — use lookup dict instead."""
    pass
def check_running_state():
    """Query Eventhouse for current running state of each asset. Returns dict {asset: True/False}."""
    kql = """
    let rv2=PiEvents|where not(Questionable) and Tag=="RV2:BTPU2BPDRUM.AG"|top 1 by Ts desc|extend asset_id="RV2_U2_Boiler",Threshold=600;
    let rv3=PiEvents|where not(Questionable) and Tag=="RV3:TXSU3TS15A.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Steam_Turbine",Threshold=6;
    let bfp=PiEvents|where not(Questionable) and Tag=="RV3:FPSU3TS24E.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Boiler_Feed_Pump_East",Threshold=20;
    union rv2,rv3,bfp
    |extend V=toreal(Value)
    |extend IsRunning=(V > Threshold)
    |project asset_id, IsRunning, V=round(V,1), Tag, Ts
    """
    try:
        rs_df = read_kusto(kql)
        status = {}
        for row in rs_df.collect():
            status[row.asset_id] = bool(row.IsRunning)
            state = "RUNNING" if row.IsRunning else "STOPPED"
            print(f"  {row.asset_id}: {state} (value={row.V}, tag={row.Tag})")
        # Default to True for any asset not found (fail open)
        for a in TARGET_ASSETS:
            if a not in status:
                status[a] = True
                print(f"  {a}: UNKNOWN (no data) — assuming RUNNING")
        return status
    except Exception as e:
        print(f"  ⚠️ Running-state check failed: {e} — assuming all RUNNING")
        return {a: True for a in TARGET_ASSETS}
print("AAKR SmartSignal Replacement initialized")
print(f"  notebook_run_id: {notebook_run_id}")
print(f"  Assets: {len(TARGET_ASSETS)}")
print(f"  Baseline: {BASELINE_DAYS} days, Memory: {MEMORY_SAMPLE_SIZE} rows/asset")
print(f"  Residual threshold: {RESIDUAL_ALERT_SIGMA}sigma")
print(f"  Exclusion buffer: {EXCLUSION_BUFFER_HOURS}h before/after events")
print("\n🔍 Checking current asset running state...")
ASSET_RUNNING = check_running_state()
ACTIVE_ASSETS = [a for a in TARGET_ASSETS if ASSET_RUNNING.get(a, True)]
STOPPED_ASSETS = [a for a in TARGET_ASSETS if not ASSET_RUNNING.get(a, True)]
if STOPPED_ASSETS:
    print(f"\n⚠️ Stopped assets (will skip scoring): {', '.join(STOPPED_ASSETS)}")
print(f"✔ Active assets for scoring: {', '.join(ACTIVE_ASSETS)}")

## Phase 1: Build Training Memory
Build a per-asset matrix of known-good sensor snapshots. Each row is a 15-minute time bin; each column is a PI tag's mean value during that bin.
**Data quality filters:**
1. Running-state filter (drum pressure, speed thresholds)
2. Anti-join GADS forced outage windows (with 6h buffer)
3. Anti-join SmartSignal observation firing windows
4. Minimum tag completeness per bin (>=80%)
5. Stratified sampling by MW load bin (all operating regimes represented)

In [ ]:
print("="*70)
print("PHASE 1: BUILDING TRAINING MEMORY")
print("="*70)
from pyspark.sql import functions as F
# --- Load PI data with asset mapping ---
# gold.fact_pi already carries the friendly asset_id; use it directly (the bridge's
# asset_id is a numeric equipment id that never matches TARGET_ASSETS -> 0 rows).
pi_data = spark.table(PI_TABLE).filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("ValueNumeric").isNotNull() &
    (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
).cache()
total_pi = pi_data.count()
print(f"✔ Loaded PI data: {total_pi:,} rows ({BASELINE_DAYS}-day window)")
# --- Running-state filter ---
running_windows_list = []
for asset, cfg in RUNNING_STATE.items():
    rs_tag, thr, op = cfg['tag'], cfg['threshold'], cfg['op']
    rs = spark.table(PI_TABLE).filter(
        (F.col("Tag") == rs_tag) &
        F.col("ValueNumeric").isNotNull() &
        (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
    ).select("Timestamp", "ValueNumeric")
    if op == '>':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") > thr).cast("int"))
    elif op == '<':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") < thr).cast("int"))
    else:
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") >= thr).cast("int"))
    w = Window.orderBy("Timestamp")
    rs = rs.withColumn("prev_state", F.lag("is_running").over(w))
    rs = rs.withColumn("state_change", (F.col("is_running") != F.coalesce(F.col("prev_state"), F.lit(-1))).cast("int"))
    rs = rs.withColumn("grp", F.sum("state_change").over(w.rowsBetween(Window.unboundedPreceding, 0)))
    windows = rs.filter(F.col("is_running") == 1).groupBy("grp").agg(
        F.min("Timestamp").alias("run_start"),
        F.max("Timestamp").alias("run_end")
    ).withColumn("asset_id", F.lit(asset)).select("asset_id", "run_start", "run_end")
    running_windows_list.append(windows)
running_windows = running_windows_list[0]
for w_df in running_windows_list[1:]:
    running_windows = running_windows.unionByName(w_df)
pi_running = pi_data.alias("p").join(
    F.broadcast(running_windows.alias("rw")),
    (F.col("p.asset_id") == F.col("rw.asset_id")) &
    (F.col("p.Timestamp").between(F.col("rw.run_start"), F.col("rw.run_end"))),
    "inner"
).select("p.*")
run_count = pi_running.count()
print(f"✔ Running-state filter: {run_count:,} rows ({100*run_count/max(total_pi,1):.1f}%)")
# --- Anti-join GADS events (with buffer) ---
buffer_expr = F.expr(f"INTERVAL {EXCLUSION_BUFFER_HOURS} HOURS")
gads_windows = spark.table("gold.fact_gads_event").filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("REAL_START_DT").isNotNull() & F.col("REAL_END_DT").isNotNull()
).select(
    F.col("asset_id").alias("ev_asset"),
    (F.col("REAL_START_DT") - buffer_expr).alias("ev_start"),
    (F.col("REAL_END_DT") + buffer_expr).alias("ev_end")
)
pi_no_gads = pi_running.join(
    F.broadcast(gads_windows),
    (pi_running.asset_id == gads_windows.ev_asset) &
    (pi_running.Timestamp.between(gads_windows.ev_start, gads_windows.ev_end)),
    "left_anti"
)
nogads_count = pi_no_gads.count()
print(f"✔ After GADS exclusion (+-{EXCLUSION_BUFFER_HOURS}h buffer): {nogads_count:,} rows")
# --- Anti-join SmartSignal firing windows ---
try:
    ss_firings = read_kusto("""
        SmartSignalObservationsRaw
        | summarize firing_start=min(timestamp), firing_end=max(timestamp)
          by asset_path, tag_name, bin(timestamp, 1h)
    """)
    ss_windows = ss_firings.filter(F.col("tag_name").isNotNull() & (F.col("tag_name") != "")).select(
        F.col("tag_name").alias("ss_tag"),
        (F.col("firing_start") - F.expr("INTERVAL 1 HOUR")).alias("ss_start"),
        (F.col("firing_end") + F.expr("INTERVAL 1 HOUR")).alias("ss_end")
    )
    pi_clean = pi_no_gads.join(
        F.broadcast(ss_windows),
        (pi_no_gads.Tag == ss_windows.ss_tag) &
        (pi_no_gads.Timestamp.between(ss_windows.ss_start, ss_windows.ss_end)),
        "left_anti"
    )
    clean_count = pi_clean.count()
    print(f"✔ After SmartSignal exclusion: {clean_count:,} rows")
except Exception as e:
    print(f"⚠️ SmartSignal exclusion skipped: {e}")
    pi_clean = pi_no_gads
    clean_count = nogads_count
pi_clean = pi_clean.cache()
# --- Bin into 15-min windows, pivot to wide per asset ---
print(f"\n⏳ Pivoting to wide format per asset...")
per_asset_memory = {}
all_memory_frames = []
for asset in TARGET_ASSETS:
    asset_pi = pi_clean.filter(F.col("asset_id") == asset)
    # Tags for this asset = tags actually present in its cleaned PI data (the bridge's
    # asset_id is numeric and cannot be filtered by the friendly asset name).
    asset_tags = sorted([r.Tag for r in asset_pi.select("Tag").distinct().collect()])
    safe_tags = [safe_col(t) for t in asset_tags]
    tag_to_safe = dict(zip(asset_tags, safe_tags))
    safe_to_tag = dict(zip(safe_tags, asset_tags))
    if not safe_tags:
        print(f"  [{asset}] no tags with data in PI - skipping")
        per_asset_memory[asset] = {'tags': [], 'safe_tags': [], 'n_tags': 0,
            'n_memory': 0, 'total_bins': 0, 'fill_map': {},
            'tag_to_safe': {}, 'safe_to_tag': {}}
        continue
    # Get MW values per bin
    mw_tag = MW_TAGS[asset]
    mw_bins = spark.table(PI_TABLE).filter(
        (F.col("Tag") == mw_tag) & F.col("ValueNumeric").isNotNull() &
        (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
    ).withColumn(
        "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
    ).groupBy("bin_start").agg(
        F.avg("ValueNumeric").alias("mw_value")
    ).withColumn(
        "mw_bin", (F.floor(F.col("mw_value") / F.lit(MW_BIN_WIDTH)) * F.lit(MW_BIN_WIDTH)).cast("int")
    )
    # Rename Tag values to safe column names before pivot
    asset_pi_safe = asset_pi.withColumn("SafeTag", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("Tag"), ":", "_"), "\\.", "_"), "-", "_"))
    # Bin sensor data
    binned = asset_pi_safe.withColumn(
        "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
    ).groupBy("bin_start").pivot("SafeTag", safe_tags).agg(F.avg("ValueNumeric"))
    # Join MW
    binned = binned.join(mw_bins.select("bin_start", "mw_value", "mw_bin"), "bin_start", "left")
    # Completeness filter
    n_tags = len(safe_tags)
    min_present = int(n_tags * MIN_COMPLETENESS)
    tag_present_expr = sum([F.when(F.col(t).isNotNull(), F.lit(1)).otherwise(F.lit(0)) for t in safe_tags], F.lit(0))
    binned = binned.withColumn("n_present", tag_present_expr).filter(F.col("n_present") >= min_present)
    total_bins = binned.count()
    if total_bins == 0:
        print(f"  [{asset}] ⚠️ No clean bins after filtering — skipping")
        per_asset_memory[asset] = {
            'tags': asset_tags, 'safe_tags': safe_tags, 'n_tags': n_tags,
            'n_memory': 0, 'total_bins': 0, 'fill_map': {},
            'tag_to_safe': tag_to_safe, 'safe_to_tag': safe_to_tag
        }
        continue
    # Stratified sampling by MW bin
    if total_bins > MEMORY_SAMPLE_SIZE:
        sample_frac = MEMORY_SAMPLE_SIZE / total_bins
        fracs = {r.mw_bin: min(1.0, sample_frac * 1.5)
                 for r in binned.select("mw_bin").distinct().collect() if r.mw_bin is not None}
        binned_sampled = binned.sampleBy("mw_bin", fractions=fracs, seed=42).limit(MEMORY_SAMPLE_SIZE)
    else:
        binned_sampled = binned
    sample_count = binned_sampled.count()
    # Fill remaining nulls with column median
    medians = binned_sampled.select([F.percentile_approx(t, 0.5).alias(t) for t in safe_tags]).collect()[0]
    fill_map = {t: float(medians[t]) if medians[t] is not None else 0.0 for t in safe_tags}
    binned_filled = binned_sampled.fillna(fill_map)
    memory_df = binned_filled.withColumn("asset_id", F.lit(asset)) \
        .withColumn("notebook_run_id", F.lit(notebook_run_id))
    all_memory_frames.append(memory_df)
    per_asset_memory[asset] = {
        'tags': asset_tags, 'safe_tags': safe_tags, 'n_tags': n_tags,
        'n_memory': sample_count, 'total_bins': total_bins, 'fill_map': fill_map,
        'tag_to_safe': tag_to_safe, 'safe_to_tag': safe_to_tag
    }
    print(f"  [{asset}] {n_tags} tags, {total_bins} clean bins -> {sample_count} memory rows")
# Save training memory
if all_memory_frames:
    for i, (asset, frame) in enumerate(zip(TARGET_ASSETS, all_memory_frames)):
        mode = "overwrite" if i == 0 else "append"
        frame.write.mode(mode).option("overwriteSchema", "true" if i == 0 else "false") \
            .option("mergeSchema", "true").format("delta").saveAsTable(MEMORY_TABLE)
print(f"\n✔ Training memory saved to {MEMORY_TABLE}")
# Save model metadata
meta_records = []
for asset, info in per_asset_memory.items():
    meta_records.append({
        'asset_id': asset, 'n_tags': info['n_tags'], 'n_memory_rows': info['n_memory'],
        'total_clean_bins': info['total_bins'], 'baseline_days': BASELINE_DAYS,
        'min_completeness': MIN_COMPLETENESS, 'exclusion_buffer_h': EXCLUSION_BUFFER_HOURS,
        'memory_sample_size': MEMORY_SAMPLE_SIZE, 'tag_list': ','.join(info['tags']),
        'trained_at': datetime.utcnow().isoformat(), 'notebook_run_id': notebook_run_id
    })
meta_df = spark.createDataFrame(pd.DataFrame(meta_records))
meta_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(MEMORY_META_TABLE)
print(f"✔ Model metadata saved to {MEMORY_META_TABLE}")

## Phase 2: AAKR Scoring Engine
For each recent 15-min bin, compute expected values and residuals for every tag using Auto-Associative Kernel Regression.
**Algorithm:**
1. Standardize the observation vector and training memory
2. Compute Euclidean distances to all memory rows
3. Apply Gaussian kernel weights (closer = higher weight)
4. Expected value = weighted average of memory rows (in original space)
5. Residual = actual - expected
6. Standardize residuals using leave-one-out calibration statistics
7. Flag anomalies where |z_residual| > threshold

In [ ]:
print("="*70)
print("PHASE 2: AAKR SCORING ENGINE")
print("="*70)
import numpy as np
from scipy.spatial.distance import cdist
def aakr_score(X_live, memory, bandwidth, tag_completeness_mask=None):
    """
    AAKR scoring: compute expected values and residuals.
    """
    if X_live.ndim != 2 or memory.ndim != 2:
        raise ValueError(f"Expected 2D arrays, got X_live={X_live.ndim}D, memory={memory.ndim}D")
    n_obs, d = X_live.shape
    N = memory.shape[0]
    if N == 0:
        return (np.full_like(X_live, np.nan), np.full_like(X_live, np.nan),
                np.zeros(n_obs), np.zeros(d), np.ones(d))
    mu = np.nanmean(memory, axis=0)
    sigma = np.nanstd(memory, axis=0)
    sigma = np.atleast_1d(sigma).copy()
    sigma[sigma < 1e-10] = 1.0
    memory_std = (memory - mu) / sigma
    expected = np.full_like(X_live, np.nan)
    effective_n = np.zeros(n_obs)
    for i in range(n_obs):
        x = X_live[i]
        if tag_completeness_mask is not None:
            mask = tag_completeness_mask[i]
            if mask.sum() < max(1, d * MIN_COMPLETENESS):
                continue
        else:
            mask = np.ones(d, dtype=bool)
        x_std = np.zeros(d)
        x_std[mask] = (x[mask] - mu[mask]) / sigma[mask]
        diffs = memory_std[:, mask] - x_std[mask]
        distances = np.sqrt(np.sum(diffs ** 2, axis=1) * (d / max(mask.sum(), 1)))
        weights = np.exp(-0.5 * (distances / max(bandwidth, 1e-10)) ** 2)
        w_sum = weights.sum()
        if w_sum < 1e-12:
            continue
        weights_norm = weights / w_sum
        effective_n[i] = 1.0 / max(np.sum(weights_norm ** 2), 1e-12)
        expected[i] = weights_norm @ memory
    residuals = X_live - expected
    return expected, residuals, effective_n, mu, sigma
def calibrate_residuals_loo(memory, bandwidth, n_loo_samples=500):
    """Leave-one-out residual calibration on training memory."""
    N, d = memory.shape
    if N < 3:
        print(f"    âš ï¸ LOO calibration: only {N} memory rows â€” using unit fallback")
        return np.zeros(d), np.ones(d)
    n_samples = min(n_loo_samples, N)
    indices = np.random.RandomState(42).choice(N, n_samples, replace=False)
    mu = np.mean(memory, axis=0)
    sigma = np.std(memory, axis=0)
    sigma = np.atleast_1d(sigma).copy()
    sigma[sigma < 1e-10] = 1.0
    memory_std = (memory - mu) / sigma
    loo_residuals = []
    for i in indices:
        x = memory[i:i+1]
        mem_without = np.delete(memory, i, axis=0)
        mem_std_without = np.delete(memory_std, i, axis=0)
        x_std = (x - mu) / sigma
        distances = np.sqrt(np.sum((mem_std_without - x_std) ** 2, axis=1))
        weights = np.exp(-0.5 * (distances / max(bandwidth, 1e-10)) ** 2)
        w_sum = weights.sum()
        if w_sum < 1e-12:
            continue
        weights_norm = weights / w_sum
        expected_val = weights_norm @ mem_without
        loo_residuals.append(x[0] - expected_val)
    if len(loo_residuals) < 2:
        print(f"    âš ï¸ LOO calibration: only {len(loo_residuals)} valid samples â€” using unit fallback")
        return np.zeros(d), np.ones(d)
    loo_arr = np.array(loo_residuals)  # shape (n_valid, d)
    residual_median = np.atleast_1d(np.median(loo_arr, axis=0))
    residual_mad = np.atleast_1d(np.median(np.abs(loo_arr - residual_median), axis=0)) * 1.4826
    # Replace near-zero MAD with std, then with 1.0
    low_mask = residual_mad < 1e-10
    if low_mask.any():
        std_vals = np.atleast_1d(np.std(loo_arr, axis=0))
        residual_mad[low_mask] = std_vals[low_mask]
    residual_mad[residual_mad < 1e-10] = 1.0
    return residual_median, residual_mad
def compute_bandwidth(memory_np):
    """Compute auto-bandwidth with guards for degenerate cases."""
    N, d = memory_np.shape
    if N < 2:
        print(f"    âš ï¸ Auto-bandwidth: only {N} rows â€” using default 1.0")
        return 1.0
    n_sample = min(500, N)
    sample_idx = np.random.RandomState(42).choice(N, n_sample, replace=False)
    mu_bw = np.mean(memory_np, axis=0)
    sigma_bw = np.std(memory_np, axis=0)
    sigma_bw = np.atleast_1d(sigma_bw).copy()
    sigma_bw[sigma_bw < 1e-10] = 1.0
    sample_std = (memory_np[sample_idx] - mu_bw) / sigma_bw
    if n_sample == 1:
        return 1.0
    dists = cdist(sample_std, sample_std, metric='euclidean')
    np.fill_diagonal(dists, np.inf)
    finite_dists = dists[dists < np.inf]
    if len(finite_dists) == 0:
        print(f"    âš ï¸ Auto-bandwidth: all distances inf/identical â€” using default 1.0")
        return 1.0
    # Remove zero distances (identical rows)
    nonzero_dists = finite_dists[finite_dists > 1e-10]
    if len(nonzero_dists) == 0:
        print(f"    âš ï¸ Auto-bandwidth: all rows identical â€” using default 1.0")
        return 1.0
    bw = float(np.median(nonzero_dists)) * 0.5
    return max(bw, 1e-6)
# --- Load training memory and score live data ---
scoring_time = datetime.utcnow()
all_scored_records = []
for asset in TARGET_ASSETS:
    info = per_asset_memory[asset]
    asset_tags = info['tags']
    safe_tags = info['safe_tags']
    safe_to_tag = info['safe_to_tag']
    n_tags = info['n_tags']
    print(f"\n{'â”€'*60}")
    print(f"Asset: {asset} ({n_tags} tags)")
    print(f"{'â”€'*60}")
    # Running-state gate: skip scoring if asset is stopped
    if not ASSET_RUNNING.get(asset, True):
        print(f"  â¸ï¸ Asset is STOPPED â€” skipping scoring")
        continue
    # Load training memory
    memory_spark = spark.table(MEMORY_TABLE).filter(F.col("asset_id") == asset)
    memory_cols = [c for c in safe_tags if c in memory_spark.columns]
    if len(memory_cols) < 3:
        print(f"  âš ï¸ Only {len(memory_cols)} tags in memory â€” skipping")
        continue
    memory_pdf = memory_spark.select(memory_cols).toPandas()
    if len(memory_pdf) == 0:
        print(f"  âš ï¸ No memory rows â€” skipping")
        continue
    memory_np = memory_pdf.values.astype(np.float64)
    # Fill NaN with column means
    col_means = np.nanmean(memory_np, axis=0)
    # Guard: if a column is ALL NaN, use 0
    col_means = np.where(np.isnan(col_means), 0.0, col_means)
    for j in range(memory_np.shape[1]):
        nan_mask = np.isnan(memory_np[:, j])
        if nan_mask.any():
            memory_np[nan_mask, j] = col_means[j]
    print(f"  Training memory: {memory_np.shape[0]} rows x {memory_np.shape[1]} tags")
    # --- Auto-bandwidth ---
    if KERNEL_BANDWIDTH == 'auto':
        bandwidth = compute_bandwidth(memory_np)
    else:
        bandwidth = float(KERNEL_BANDWIDTH)
    print(f"  Kernel bandwidth: {bandwidth:.4f}")
    # --- LOO residual calibration ---
    print(f"  â³ Calibrating residuals (LOO)...")
    res_median, res_mad = calibrate_residuals_loo(memory_np, bandwidth, n_loo_samples=500)
    print(f"  âœ” LOO calibration done (median MAD: {float(np.median(res_mad)):.4f})")
    # --- Pull live data from Eventhouse ---
    original_tags = [safe_to_tag.get(c, c) for c in memory_cols]
    tag_filter = "'" + "','".join(original_tags) + "'"
    mw_tag = MW_TAGS[asset]
    try:
        live_pi = read_kusto(f"""
            PiEvents
            | where Ts > ago({SCORING_LOOKBACK_HOURS}h)
            | where not(Questionable)
            | where Tag in ({tag_filter}) or Tag == '{mw_tag}'
            | summarize Value = avg(toreal(Value)) by bin(Ts, {BIN_SECONDS}s), Tag
            | project Tag, Timestamp = Ts, ValueNumeric = Value
        """).filter(F.col("ValueNumeric").isNotNull() & ~F.isnan(F.col("ValueNumeric")))
    except Exception as e:
        print(f"  âš ï¸ Eventhouse query failed: {e}")
        continue
    live_count = live_pi.count()
    if live_count == 0:
        print(f"  âš ï¸ No live data â€” skipping")
        continue
    # Pivot live data with safe column names
    live_pi_safe = live_pi.filter(F.col("Tag").isin(original_tags)).withColumn(
        "SafeTag", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("Tag"), ":", "_"), "\\.", "_"), "-", "_")
    )
    live_wide = live_pi_safe.withColumn(
        "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
    ).groupBy("bin_start").pivot("SafeTag", memory_cols).agg(
        F.avg("ValueNumeric")
    ).orderBy("bin_start")
    live_pdf = live_wide.toPandas()
    if len(live_pdf) == 0:
        print(f"  âš ï¸ No bins after pivot â€” skipping")
        continue
    bin_starts = live_pdf['bin_start'].values
    live_np = live_pdf[memory_cols].values.astype(np.float64)
    # Build completeness mask and fill NaNs
    completeness_mask = ~np.isnan(live_np)
    for j in range(live_np.shape[1]):
        nan_mask = np.isnan(live_np[:, j])
        if nan_mask.any():
            live_np[nan_mask, j] = col_means[j]
    print(f"  Live data: {live_np.shape[0]} bins x {live_np.shape[1]} tags")
    # --- Run AAKR ---
    expected, residuals, eff_n, mu, sigma = aakr_score(
        live_np, memory_np, bandwidth, completeness_mask
    )
    # Standardize residuals using LOO calibration
    z_residuals = (residuals - res_median) / res_mad
    z_residuals = np.clip(z_residuals, -100, 100)
    is_anomaly = np.abs(z_residuals) > RESIDUAL_ALERT_SIGMA
    # --- Collect results (skip NaN rows) ---
    n_anomalies = 0
    for i in range(live_np.shape[0]):
        if np.any(np.isnan(expected[i])):
            continue
        for j, safe_tag in enumerate(memory_cols):
            original_tag = safe_to_tag.get(safe_tag, safe_tag)
            z_val = z_residuals[i, j]
            if np.isnan(z_val) or np.isinf(z_val):
                continue
            record = {
                'asset_id': asset, 'bin_start': str(bin_starts[i]),
                'tag': original_tag,
                'actual_value': float(live_np[i, j]),
                'expected_value': float(expected[i, j]),
                'residual': float(residuals[i, j]),
                'z_residual': float(z_val),
                'is_anomaly': bool(is_anomaly[i, j]),
                'effective_n': float(eff_n[i]),
                'bandwidth': bandwidth,
                'notebook_run_id': notebook_run_id
            }
            all_scored_records.append(record)
            if record['is_anomaly']:
                n_anomalies += 1
    anomaly_tags = int(np.sum(np.any(is_anomaly, axis=0)))
    print(f"  âœ” Scored {live_np.shape[0]} bins: {n_anomalies} tag-bin anomalies across {anomaly_tags} tags")
    valid_n = eff_n[eff_n > 0]
    if len(valid_n) > 0:
        print(f"  Effective N range: {np.min(valid_n):.0f} â€“ {np.max(valid_n):.0f}")
    else:
        print(f"  âš ï¸ No valid scores (all effective N = 0)")
# --- Build scored DataFrame ---
if all_scored_records:
    scored_df = spark.createDataFrame(pd.DataFrame(all_scored_records))
    scored_df = scored_df.withColumn("scored_at", F.current_timestamp())
    scored_df.cache()
    total_scored = scored_df.count()
    total_anomalies = scored_df.filter(F.col("is_anomaly")).count()
    print(f"\n{'='*60}")
    print(f"AAKR SCORING COMPLETE")
    print(f"  Total tag-bin scores: {total_scored:,}")
    print(f"  Anomalies: {total_anomalies} ({100*total_anomalies/max(total_scored,1):.2f}%)")
    print(f"{'='*60}")
    print(f"\nTop anomalies by |z_residual|:")
    scored_df.filter(F.col("is_anomaly")).orderBy(F.desc(F.abs(F.col("z_residual")))) \
        .select("asset_id", "tag", "bin_start", "actual_value", "expected_value",
                "residual", "z_residual", "effective_n") \
        .show(20, truncate=False)
else:
    scored_df = None
    print("\nâš ï¸ No scores produced â€” check Eventhouse feed")

## Phase 3: Threshold Calibration via SmartSignal
Validate AAKR detections against SmartSignal incidents (ground truth). For the 60 overlapping tags, compute precision/recall at various thresholds to find the optimal alert sensitivity.

In [ ]:
print("="*70)
print("PHASE 3: THRESHOLD CALIBRATION vs SMARTSIGNAL")
print("="*70)
if scored_df is None:
    print("⚠️ No scored data — skipping calibration")
else:
    # Load SmartSignal incidents from Eventhouse
    try:
        ss_incidents = read_kusto("""
            SmartSignalIncidentsRaw
            | where isnotempty(tag_name)
            | summarize
                ss_start = min(first_occurred),
                ss_end = max(last_occurred),
                n_incidents = count(),
                max_severity = max(priority)
              by asset_path, tag_name
        """)
        ss_count = ss_incidents.count()
        print(f"✔ Loaded {ss_count} SmartSignal incident groups (tag × asset)")
        # Find overlapping tags between AAKR and SmartSignal
        aakr_tags = scored_df.select("tag").distinct()
        ss_tags = ss_incidents.select(F.col("tag_name").alias("tag")).distinct()
        overlap = aakr_tags.join(ss_tags, "tag", "inner")
        n_overlap = overlap.count()
        print(f"  Overlapping tags: {n_overlap}")
        if n_overlap > 0:
            # Compare at various thresholds
            thresholds = [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
            print(f"\n{'Threshold':>10} {'Anom Tags':>10} {'SS Tags':>10} {'Overlap':>10} {'Precision':>10} {'Recall':>10}")
            print("─" * 65)
            ss_alert_tags = set(r.tag for r in ss_incidents.select(F.col("tag_name").alias("tag")).distinct().collect())
            for thr in thresholds:
                aakr_alerts = set(r.tag for r in
                    scored_df.filter(F.abs(F.col("z_residual")) > thr)
                        .select("tag").distinct().collect())
                tp = len(aakr_alerts & ss_alert_tags)
                fp = len(aakr_alerts - ss_alert_tags)
                fn = len(ss_alert_tags - aakr_alerts)
                precision = tp / max(tp + fp, 1)
                recall = tp / max(tp + fn, 1)
                print(f"{thr:>10.1f} {len(aakr_alerts):>10} {len(ss_alert_tags):>10} {tp:>10} {precision:>10.3f} {recall:>10.3f}")
            # Select best threshold (F1)
            best_thr = 3.0
            best_f1 = 0.0
            for thr in thresholds:
                aakr_alerts = set(r.tag for r in
                    scored_df.filter(F.abs(F.col("z_residual")) > thr)
                        .select("tag").distinct().collect())
                tp = len(aakr_alerts & ss_alert_tags)
                fp = len(aakr_alerts - ss_alert_tags)
                fn = len(ss_alert_tags - aakr_alerts)
                p = tp / max(tp + fp, 1)
                r = tp / max(tp + fn, 1)
                f1 = 2 * p * r / max(p + r, 1e-10)
                if f1 > best_f1:
                    best_f1 = f1
                    best_thr = thr
            print(f"\n✔ Best threshold by F1: {best_thr} (F1 = {best_f1:.3f})")
            CALIBRATED_THRESHOLD = best_thr
        else:
            print("  ⚠️ No tag overlap — using default threshold")
            CALIBRATED_THRESHOLD = RESIDUAL_ALERT_SIGMA
    except Exception as e:
        print(f"⚠️ SmartSignal calibration skipped: {e}")
        CALIBRATED_THRESHOLD = RESIDUAL_ALERT_SIGMA
    print(f"\n→ Using threshold: |z| > {CALIBRATED_THRESHOLD}")

## Phase 4: Alert Generation + Persistence
Collapse AAKR anomalous bins into episodes, assign severity, and persist to the shared gold tables.

In [ ]:
print("="*70)
print("PHASE 4: ALERT GENERATION + PERSISTENCE")
print("="*70)
if scored_df is None:
    print(" generation")
else:
    # Apply calibrated threshold
    alerts = scored_df.filter(F.abs(F.col("z_residual")) > CALIBRATED_THRESHOLD)
    n_alerts = alerts.count()
    print(f"Anomalous tag-bins at |z| > {CALIBRATED_THRESHOLD}: {n_alerts}")
    if n_alerts > 0:
        # Collapse into episodes: group consecutive anomalous bins per (asset, tag)
        from pyspark.sql import Window as W
        alerts_ordered = alerts.withColumn(
            "bin_ts", F.to_timestamp("bin_start")
        ).orderBy("asset_id", "tag", "bin_ts")
        w = W.partitionBy("asset_id", "tag").orderBy("bin_ts")
        alerts_ordered = alerts_ordered.withColumn(
            "prev_bin", F.lag("bin_ts").over(w)
        ).withColumn(
            "gap_seconds",
            F.when(F.col("prev_bin").isNull(), F.lit(99999))
             .otherwise((F.col("bin_ts").cast("long") - F.col("prev_bin").cast("long")))
        ).withColumn(
            "new_episode", F.when(F.col("gap_seconds") > BIN_SECONDS * 2, F.lit(1)).otherwise(F.lit(0))
        ).withColumn(
            "episode_id", F.sum("new_episode").over(w.rowsBetween(W.unboundedPreceding, 0))
        )
        episodes = alerts_ordered.groupBy("asset_id", "tag", "episode_id").agg(
            F.min("bin_ts").alias("episode_start"),
            F.max("bin_ts").alias("episode_end"),
            F.count("*").alias("n_bins"),
            F.max(F.abs(F.col("z_residual"))).alias("max_z"),
            F.avg(F.col("z_residual")).alias("mean_z"),
            F.avg("actual_value").alias("mean_actual"),
            F.avg("expected_value").alias("mean_expected"),
            F.avg("residual").alias("mean_residual")
        ).withColumn(
            "duration_minutes", (F.col("episode_end").cast("long") - F.col("episode_start").cast("long")) / 60
        ).withColumn(
            "severity", F.when(F.col("max_z") > 6.0, "CRITICAL")
                         .when(F.col("max_z") > 4.5, "HIGH")
                         .when(F.col("max_z") > 3.5, "MEDIUM")
                         .otherwise("LOW")
        ).withColumn(
            "algorithm", F.lit("AAKR")
        ).withColumn(
            "model_version", F.lit(notebook_run_id)
        ).withColumn(
            "detected_at", F.current_timestamp()
        )
        n_episodes = episodes.count()
        print(f"")
        episodes.select("asset_id", "tag", "episode_start", "episode_end",
                        "duration_minutes", "n_bins", "max_z", "severity") \
            .orderBy(F.desc("max_z")).show(20, truncate=False)
        # --- Persist scored data ---
        scored_df.write.mode("append").option("mergeSchema", "true") \
            .format("delta").saveAsTable(SCORED_TABLE)
        print(f"")
        # --- Persist episodes ---
        episodes.write.mode("overwrite").option("overwriteSchema", "true") \
            .format("delta").saveAsTable(EPISODES_TABLE)
        print(f"")
                # --- Persist to unified watchlist (1 row per asset, top 3 tags) ---
        bridge_desc = spark.table("gold.bridge_pi_tag_to_asset") \
            .select("Tag", "tag_description").dropDuplicates(["Tag"]).toPandas()
        tag_desc_map = dict(zip(bridge_desc['Tag'], bridge_desc['tag_description']))
        epi_pdf = episodes.orderBy(F.desc("max_z")).toPandas()
        wl_rows = []
        for asset_id, grp in epi_pdf.groupby("asset_id"):
            top3 = grp.nlargest(3, "max_z")
            worst_z = top3["max_z"].max()
            severity = "CRITICAL" if worst_z > 6.0 else "HIGH" if worst_z > 4.5 else "MEDIUM" if worst_z > 3.5 else "LOW"
            friendly = [tag_desc_map.get(t, t).title() for t in top3["tag"]]
            watch_str = ", ".join(dict.fromkeys(friendly))
            top_tag = top3.iloc[0]["tag"]
            wl_rows.append({
                "model_name": "AAKR_SmartSignal",
                "scoring_date": str(datetime.now().date()),
                "asset_id": asset_id,
                "feature": "aakr_summary",
                "tag_name": top_tag,
                "descriptor": tag_desc_map.get(top_tag, top_tag).title(),
                "engineering_units": None,
                "current_value": float(top3.iloc[0]["mean_actual"]),
                "baseline_mean": float(top3.iloc[0]["mean_expected"]),
                "baseline_std": float(top3.iloc[0]["mean_residual"]),
                "normal_range_low": None, "normal_range_high": None,
                "risk_contribution": float(worst_z),
                "trend_direction": None, "trend_slope_per_day": None,
                "recommended_action": severity,
                "recommendation_text": f"Watch {watch_str}. Worst anomaly z={worst_z:.1f}.",
                "watch_horizon_days": 14,
                "model_run_timestamp": datetime.now(),
                "notebook_run_id": notebook_run_id
            })
        import pandas as pd
        wl_df = spark.createDataFrame(pd.DataFrame(wl_rows))
        wl_df = wl_df.withColumn("watch_horizon_days", F.col("watch_horizon_days").cast("int"))
        wl_df.write.mode("append").format("delta").saveAsTable("ml.watchlist")
        print(f"Anomalous tag-bins at |z| > {CALIBRATED_THRESHOLD}: {n_alerts}")
    if n_alerts > 0:
        # Collapse into episodes: group consecutive anomalous bins per (asset, tag)
        from pyspark.sql import Window as W
        alerts_ordered = alerts.withColumn(
            "bin_ts", F.to_timestamp("bin_start")
        ).orderBy("asset_id", "tag", "bin_ts")
        w = W.partitionBy("asset_id", "tag").orderBy("bin_ts")
        alerts_ordered = alerts_ordered.withColumn(
            "prev_bin", F.lag("bin_ts").over(w)
        ).withColumn(
            "gap_seconds",
            F.when(F.col("prev_bin").isNull(), F.lit(99999))
             .otherwise((F.col("bin_ts").cast("long") - F.col("prev_bin").cast("long")))
        ).withColumn(
            "new_episode", F.when(F.col("gap_seconds") > BIN_SECONDS * 2, F.lit(1)).otherwise(F.lit(0))
        ).withColumn(
            "episode_id", F.sum("new_episode").over(w.rowsBetween(W.unboundedPreceding, 0))
        )
        episodes = alerts_ordered.groupBy("asset_id", "tag", "episode_id").agg(
            F.min("bin_ts").alias("episode_start"),
            F.max("bin_ts").alias("episode_end"),
            F.count("*").alias("n_bins"),
            F.max(F.abs(F.col("z_residual"))).alias("max_z"),
            F.avg(F.col("z_residual")).alias("mean_z"),
            F.avg("actual_value").alias("mean_actual"),
            F.avg("expected_value").alias("mean_expected"),
            F.avg("residual").alias("mean_residual")
        ).withColumn(
            "duration_minutes", (F.col("episode_end").cast("long") - F.col("episode_start").cast("long")) / 60
        ).withColumn(
            "severity", F.when(F.col("max_z") > 6.0, "CRITICAL")
                         .when(F.col("max_z") > 4.5, "HIGH")
                         .when(F.col("max_z") > 3.5, "MEDIUM")
                         .otherwise("LOW")
        ).withColumn(
            "algorithm", F.lit("AAKR")
        ).withColumn(
            "model_version", F.lit(notebook_run_id)
        ).withColumn(
            "detected_at", F.current_timestamp()
        )
        n_episodes = episodes.count()
        print(f"")
        episodes.select("asset_id", "tag", "episode_start", "episode_end",
                        "duration_minutes", "n_bins", "max_z", "severity") \
            .orderBy(F.desc("max_z")).show(20, truncate=False)
        # --- Persist scored data ---
        scored_df.write.mode("append").option("mergeSchema", "true") \
            .format("delta").saveAsTable(SCORED_TABLE)
        print(f"")
        # --- Persist episodes ---
        episodes.write.mode("overwrite").option("overwriteSchema", "true") \
            .format("delta").saveAsTable(EPISODES_TABLE)
        print(f"")
        # --- Summary stats ---
        print(f"\nSeverity distribution:")
        episodes.groupBy("severity").agg(
            F.count("*").alias("episodes"),
            F.countDistinct("tag").alias("unique_tags"),
            F.countDistinct("asset_id").alias("assets")
        ).show()
    else:
        print("hy")
        # Still save scored data for audit trail
        scored_df.write.mode("append").option("mergeSchema", "true") \
            .format("delta").saveAsTable(SCORED_TABLE)
        print(f"")
        # Write NORMAL watchlist rows so stale CRITICAL entries get superseded
        bridge_desc = spark.table("gold.bridge_pi_tag_to_asset") \
            .select("Tag", "tag_description", "asset_id").dropDuplicates(["asset_id"])
        normal_rows = []
        for row in bridge_desc.collect():
            normal_rows.append({
                "model_name": "AAKR_SmartSignal",
                "scoring_date": str(datetime.now().date()),
                "asset_id": row["asset_id"],
                "feature": "aakr_summary",
                "tag_name": None,
                "descriptor": "All sensors normal",
                "engineering_units": None,
                "current_value": None,
                "baseline_mean": None,
                "baseline_std": None,
                "normal_range_low": None, "normal_range_high": None,
                "risk_contribution": 0.0,
                "trend_direction": None, "trend_slope_per_day": None,
                "recommended_action": "LOW",
                "recommendation_text": "AAKR detected no anomalies. All sensors within expected range.",
                "watch_horizon_days": 14,
                "model_run_timestamp": datetime.now(),
                "notebook_run_id": notebook_run_id
            })
        if normal_rows:
            import pandas as pd
            wl_normal = spark.createDataFrame(pd.DataFrame(normal_rows))
            wl_normal = wl_normal.withColumn("watch_horizon_days", F.col("watch_horizon_days").cast("int"))
            wl_normal.write.mode("append").format("delta").saveAsTable("ml.watchlist")
            print(f"Wrote {len(normal_rows)} NORMAL watchlist rows")

## Phase 5: Validation & Health Summary
Side-by-side comparison of AAKR vs SmartSignal detections, plus per-asset health scores.

In [ ]:
print("="*70)
print("PHASE 5: VALIDATION & HEALTH SUMMARY")
print("="*70)
from pyspark.sql import Window as W
# --- Per-asset health score ---
if scored_df is not None:
    health_records = []
    for asset in TARGET_ASSETS:
        asset_scored = scored_df.filter(F.col("asset_id") == asset)
        total_tag_bins = asset_scored.count()
        if total_tag_bins == 0:
            continue
        anom_count = asset_scored.filter(F.abs(F.col("z_residual")) > CALIBRATED_THRESHOLD).count()
        anom_pct = 100.0 * anom_count / total_tag_bins
        # Health score: 100 = perfectly normal, 0 = all anomalous
        health_score = max(0.0, 100.0 - anom_pct * 10)  # 10% anomaly → 0 health
        stats = asset_scored.agg(
            F.avg(F.abs(F.col("z_residual"))).alias("mean_abs_z"),
            F.max(F.abs(F.col("z_residual"))).alias("max_abs_z"),
            F.avg("effective_n").alias("mean_eff_n"),
            F.countDistinct("tag").alias("n_tags")
        ).collect()[0]
        print(f"\n{'─'*50}")
        print(f"  Asset: {asset}")
        print(f"  Health Score: {health_score:.1f} / 100")
        print(f"  Tag-bins scored: {total_tag_bins:,}")
        print(f"  Anomalous: {anom_count} ({anom_pct:.2f}%)")
        print(f"  Mean |z|: {stats.mean_abs_z:.3f}")
        print(f"  Max |z|: {stats.max_abs_z:.3f}")
        print(f"  Mean effective N: {stats.mean_eff_n:.0f}")
        print(f"  Tags: {stats.n_tags}")
        health_records.append({
            'asset_id': asset, 'health_score': round(health_score, 1),
            'total_tag_bins': total_tag_bins, 'anomalous_tag_bins': anom_count,
            'anomaly_pct': round(anom_pct, 2),
            'mean_abs_z': round(float(stats.mean_abs_z), 4),
            'max_abs_z': round(float(stats.max_abs_z), 4),
            'mean_effective_n': round(float(stats.mean_eff_n), 1),
            'n_tags': int(stats.n_tags),
            'threshold': CALIBRATED_THRESHOLD,
            'scored_at': datetime.utcnow().isoformat(),
            'notebook_run_id': notebook_run_id
        })
    # --- Save health summary ---
    if health_records:
        health_df = spark.createDataFrame(pd.DataFrame(health_records))
        health_df.write.mode("append").option("mergeSchema", "true") \
            .format("delta").saveAsTable(HEALTH_TABLE)
        print(f"\n✔ Health summary saved to {HEALTH_TABLE}")
    # --- SmartSignal comparison (if incidents available) ---
    try:
        ss_incidents = read_kusto("""
            SmartSignalIncidentsRaw
            | where isnotempty(tag_name)
            | summarize n_incidents = count(), max_severity = max(priority)
              by tag_name
        """)
        scored_tags_summary = scored_df.groupBy("tag").agg(
            F.max(F.abs(F.col("z_residual"))).alias("aakr_max_z"),
            F.sum(F.when(F.abs(F.col("z_residual")) > CALIBRATED_THRESHOLD, 1).otherwise(0)).alias("aakr_anomalies")
        )
        comparison = scored_tags_summary.join(
            ss_incidents.select(F.col("tag_name").alias("tag"), "n_incidents", "max_severity"),
            "tag", "full"
        ).fillna(0)
        print(f"\n{'='*60}")
        print("AAKR vs SmartSignal — Tag-Level Comparison")
        print(f"{'='*60}")
        both_alert = comparison.filter((F.col("aakr_anomalies") > 0) & (F.col("n_incidents") > 0)).count()
        aakr_only = comparison.filter((F.col("aakr_anomalies") > 0) & (F.col("n_incidents") == 0)).count()
        ss_only = comparison.filter((F.col("aakr_anomalies") == 0) & (F.col("n_incidents") > 0)).count()
        neither = comparison.filter((F.col("aakr_anomalies") == 0) & (F.col("n_incidents") == 0)).count()
        print(f"  Both alert: {both_alert}")
        print(f"  AAKR only: {aakr_only}")
        print(f"  SmartSignal only: {ss_only}")
        print(f"  Neither: {neither}")
        if both_alert + aakr_only + ss_only > 0:
            agreement = both_alert / (both_alert + aakr_only + ss_only)
            print(f"  Agreement (Jaccard): {agreement:.3f}")
        comparison.filter(
            (F.col("aakr_anomalies") > 0) | (F.col("n_incidents") > 0)
        ).orderBy(F.desc("aakr_max_z")).show(30, truncate=False)
    except Exception as e:
        print(f"\n⚠️ SmartSignal comparison skipped: {e}")
    # --- Final summary ---
    print(f"\n{'='*70}")
    print("AAKR NOTEBOOK RUN COMPLETE")
    print(f"  Run ID: {notebook_run_id}")
    print(f"  Scored at: {scoring_time.isoformat()}")
    print(f"  Threshold: |z| > {CALIBRATED_THRESHOLD}")
    print(f"  Tables written:")
    print(f"    - {MEMORY_TABLE} (training memory)")
    print(f"    - {MEMORY_META_TABLE} (model metadata)")
    print(f"    - {SCORED_TABLE} (all scores)")
    print(f"    - {EPISODES_TABLE} (anomaly episodes)")
    print(f"    - {HEALTH_TABLE} (health summary)")
    print(f"{'='*70}")
else:
    print("⚠️ No scored data available — run Phase 2 first")